In [1]:
import numpy as np
import math
from PCAClass import PCABounding, Best_Fit_CPP
import open3d as o3d
import time
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

#import cupy as np


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
#Potential-Field Displacement Visualization
def create_arrow(start, end, radius=0.5, resolution=20):
    scaleing_factor=2
    direction = end - start
    length = np.linalg.norm(direction)
    
    if length> 10:  # way beyond your max expected delta
        print(f"Unexpected large delta: {length:.2f} mm")
        print(f"start: {start}, end: {end}")

    if length < 1e-6:
        return None  # skip zero-length arrows

    arrow = o3d.geometry.TriangleMesh.create_arrow(
        cylinder_radius=radius,
        cone_radius=radius * 1.5,
        cylinder_height=np.linalg.norm(end - start)*scaleing_factor/2,
        cone_height=np.linalg.norm(end - start)*scaleing_factor/2,
        resolution=resolution,
        cylinder_split=4,
        cone_split=1,
    )

    direction_norm = direction / length

    # Compute rotation matrix from [0, 0, 1] (Z-axis) to direction with Rodrigues formula
    # z_axis = np.array([0.0, 0.0, 1.0])
    # v = np.cross(z_axis, direction_norm)
    # c=np.clip(np.dot(z_axis, direction_norm), -1.0, 1.0)
    # if np.linalg.norm(v) < 1e-6:
    #     R = np.eye(3) if c > 0 else -np.eye(3)  # aligned or opposite
    # else:
    #     vx = np.array([
    #         [0, -v[2], v[1]],
    #         [v[2], 0, -v[0]],
    #         [-v[1], v[0], 0]
    #     ])
    #     R = np.eye(3) + vx + vx @ vx * ((1 - c) / (np.linalg.norm(v) ** 2))

    z_axis = np.array([0, 0, 1])
    axis = np.cross(z_axis, direction_norm)
    if np.linalg.norm(axis) < 1e-6:
        R = np.eye(3) if np.dot(z_axis, direction_norm) > 0 else -np.eye(3)
    else:
        axis = axis / np.linalg.norm(axis)
        angle = np.arccos(np.clip(np.dot(z_axis, direction_norm), -1.0, 1.0))
        R = o3d.geometry.get_rotation_matrix_from_axis_angle(axis * angle)

    arrow.rotate(R, center=(0, 0, 0))
    arrow.translate(start)
    

    return arrow
    
def coverage_rate(segment):
    #Calculate Coverage-Score for learning rate:
    matches_Red = np.sum(np.all(segment.colors == np.array([1,0,0]), axis=1))
    matches_Blue=np.sum(np.all(segment.colors== np.array([0,0,1]), axis=1))
    matches_Green=segment.colors.shape[0]-(matches_Blue+matches_Red)
    totalsum=matches_Blue+matches_Green+matches_Red
    print(f"Red: {matches_Red/totalsum*100}%, Green: {matches_Green/totalsum*100}%, Blue: {matches_Blue/totalsum*100}%")

    return 1-matches_Red/totalsum

In [3]:
#Standard Potential Field

def Potential_Field(segment: Best_Fit_CPP, On_surface_passes):
        Off_surface_passes=[]
        for pass_set in On_surface_passes:
            new_pass_set=[]
            for i in pass_set:
                #Neighborhood is 2r
                Neighborhood_indices = np.array(segment.kd_tree.query_ball_point(i, 2*32))
                red_indices_mask= (segment.colors[Neighborhood_indices] == [1, 0, 0]).all(axis=1)
                if not np.any(red_indices_mask):
                    new_pass_set.append(i)
                else:
                    sigma=10
                    red_dist=segment.points[Neighborhood_indices[red_indices_mask]] - i 
                    red_weights = np.exp(-np.linalg.norm(red_dist, axis=1)  / sigma )
                    red_cum_direction = np.sum(red_dist * red_weights[:, np.newaxis], axis=0)
                    red_cum_norm=np.linalg.norm(red_cum_direction)
                    #print(red_cum_direction)
                    if red_cum_norm<3:
                        new_pass_set.append(i+red_cum_direction)
                    else:
                        new_pass_set.append(i+3*red_cum_direction/red_cum_norm)
            Off_surface_passes.append(np.vstack(new_pass_set))
        return Off_surface_passes


In [4]:
# Laplacian Regularization Potential Field

def LapReg_Potential_Field(segment: Best_Fit_CPP, On_surface_passes, smoothing_coeff):
        Off_surface_passes=[]
        for pass_set in On_surface_passes:
            new_pass_set=[]
            for i in pass_set:
                #Neighborhood is 2r
                Neighborhood_indices = np.array(segment.kd_tree.query_ball_point(i, 2*32))
                red_indices_mask= (segment.colors[Neighborhood_indices] == [1, 0, 0]).all(axis=1)
                if not np.any(red_indices_mask):
                    new_pass_set.append(i)
                else:
                    sigma=10
                    red_dist=segment.points[Neighborhood_indices[red_indices_mask]] - i 
                    red_weights = np.exp(-np.linalg.norm(red_dist, axis=1)  / sigma )
                    red_cum_direction = np.sum(red_dist * red_weights[:, np.newaxis], axis=0)
                    red_cum_norm=np.linalg.norm(red_cum_direction)
                    #print(red_cum_direction)
                    if red_cum_norm<3:
                        new_pass_set.append(i+red_cum_direction)
                    else:
                        new_pass_set.append(i+3*red_cum_direction/red_cum_norm)

            smoothed_pass_set=np.copy(new_pass_set)
            for i in range(1,len(new_pass_set)-1):
                 laplacian= 0.5*(new_pass_set[i-1]+new_pass_set[i+1]) - new_pass_set[i]
                 smoothed_pass_set[i] += laplacian*smoothing_coeff
            
            Off_surface_passes.append(np.vstack(smoothed_pass_set))
                 
        return Off_surface_passes


In [ ]:
#Converage Informed Learning Rate modulation (with laplacian smoothing)

def Learned_LapReg_Potential_Field(segment: Best_Fit_CPP, On_surface_passes, smoothing_coeff, coverage):
        Off_surface_passes=[]
        for pass_set in On_surface_passes:
            new_pass_set=[]
            for i in pass_set:
                #Neighborhood is 2r
                Neighborhood_indices = np.array(segment.kd_tree.query_ball_point(i, 2*32))
                red_indices_mask= (segment.colors[Neighborhood_indices] == [1, 0, 0]).all(axis=1)
                if not np.any(red_indices_mask):
                    new_pass_set.append(i)
                else:
                    sigma=10
                    red_dist=segment.points[Neighborhood_indices[red_indices_mask]] - i 
                    red_weights = np.exp(-np.linalg.norm(red_dist, axis=1)  / sigma )
                    red_cum_direction = np.sum(red_dist * red_weights[:, np.newaxis], axis=0)
                    red_cum_norm=np.linalg.norm(red_cum_direction)

                    scaling_factor= 10*()
                    #print(red_cum_direction)
                    if red_cum_norm<3:
                        new_pass_set.append(i+red_cum_direction)
                    else:
                        new_pass_set.append(i+3*red_cum_direction/red_cum_norm)

            smoothed_pass_set=np.copy(new_pass_set)
            for i in range(1,len(new_pass_set)-1):
                 laplacian= 0.5*(new_pass_set[i-1]+new_pass_set[i+1]) - new_pass_set[i]
                 smoothed_pass_set[i] += laplacian*smoothing_coeff
            
            Off_surface_passes.append(np.vstack(smoothed_pass_set))
                 
        return Off_surface_passes

In [ ]:
#Toy Example Construction
segment=Best_Fit_CPP("plane_segments\Hyper_meshed_noise.stl")
path1=np.linspace([75,1,0],[75,400,0],51)
path2=np.linspace([125,1,0],[125,400,0],51)
path3=np.linspace([175,1,0],[175,400,0],51)


#Psuedo Color-Map
# colormaskRed= (segment.points[:,0] <=50) | (segment.points[:,0]>=200)
# colormaskBlue= (segment.points[:,0] >=100) & (segment.points[:,0]<=150)
# segment.colors= np.full((segment.points.shape[0], 3), [0,1,0])
# segment.colors[colormaskBlue]=[0,0,1]
# segment.colors[colormaskRed]=[1,0,0]
# test_path=o3d.geometry.PointCloud()
# points_combo=np.vstack([path1,path2,path3])+np.array([0,0,10])
# test_path.points=o3d.utility.Vector3dVector(points_combo)
# test_path.colors=o3d.utility.Vector3dVector(np.full((points_combo.shape[0],3), [1,.6,0]))
# segment.mesh.vertex_colors = o3d.utility.Vector3dVector(segment.colors)
# o3d.visualization.draw_geometries([segment.mesh,test_path])

#Initialization
segment.color=[np.asarray([0, 0.3, 0]), np.asarray([0, 0.5, 0.5]),np.asarray([0, 0, 1])]
segment.passes=[path1,path2,path3]
#segment.color=[np.asarray([0, 0.3, 0]), np.asarray([0,0.5,1]), np.asarray([0, 0.5, 0.5]),np.asarray([0, 0, 1])]
#segment.passes=[path1,path2,path3, path4]
segment.passes_colors=[col*np.ones((len(pas), 1)) for col, pas in zip(segment.color, segment.passes)]
                       
#Toy Example                       
redundant, On_surface, On_surface_color=segment.local_scanned_area(Redundancy=False, Elimination=False)

pass_points=o3d.geometry.PointCloud()
pass_points.points=o3d.utility.Vector3dVector(np.vstack(On_surface))
pass_points.colors=o3d.utility.Vector3dVector(np.vstack(On_surface_color))
revised_points=np.vstack(On_surface)
segment.mesh.vertex_colors = o3d.utility.Vector3dVector(segment.colors)
#o3d.visualization.draw_geometries([segment.mesh,pass_points],mesh_show_back_face=True)
smoothing_coeff=0.5

for i in range(10):
    old_points=revised_points.copy()

    #segment.passes=Potential_Field(segment, On_surface)
    segment.passes=LapReg_Potential_Field(segment, On_surface, smoothing_coeff)

    redundant, On_surface, On_surface_color=segment.local_scanned_area(Redundancy=False, Elimination=False)
    pass_points=o3d.geometry.PointCloud()
    pass_points.points=o3d.utility.Vector3dVector(np.vstack(On_surface))
    pass_points.colors=o3d.utility.Vector3dVector(np.vstack(On_surface_color))
    revised_points=np.vstack(On_surface)
    segment.mesh.vertex_colors = o3d.utility.Vector3dVector(segment.colors)
    #o3d.visualization.draw_geometries([segment.mesh,pass_points],mesh_show_back_face=True)

    arrows = [segment.mesh, pass_points]
    for start, end in zip(old_points, revised_points):
        if np.linalg.norm(end - start) > 1e-3:  # Skip nearly-zero shifts
            arrows.append(create_arrow(start, end))
    o3d.visualization.draw_geometries(arrows)




In [5]:
# Full Example Construction (Adding coverage informed learning rate modulation)
segment=Best_Fit_CPP(r"C:\Users\jonas\NDIBARC\pointcloudcpp\plane_segments\Nose_Cone.stl")
segment.mesh.compute_adjacency_list()
segment.mesh.compute_triangle_normals()
segment.mesh.compute_vertex_normals()
segment.mesh.paint_uniform_color([0.5, 0.5, 0.5])
segment.boundary_edge_calculations()
segment.edge_fitter(segment.edges)
Probe_width=64
slice_resolution=5 # mm per sampled point
segment.scan_information(Probe_width, slice_resolution)
segment.shift_direction()
shift_vec=segment.offset_one*segment.secondary_axis
segment.edge1_CP=segment.edge1_CP - segment.offset_dir*shift_vec
segment.edge2_CP=segment.edge2_CP + segment.offset_dir*shift_vec
passes,colors=segment.line_interpolator(10)
adjusted_lines = np.vstack(passes)+ np.array([0, 0, 10])
#Path visualization
trial=o3d.geometry.PointCloud()
trial.points=o3d.utility.Vector3dVector(adjusted_lines)
trial.colors=o3d.utility.Vector3dVector(np.vstack(colors))
#segment.fancy_viz([segment.mesh, trial, segment.bounding_box])
o3d.visualization.draw_geometries([segment.mesh,segment.bounding_box, trial],mesh_show_back_face=True)


redundant, On_surface, On_surface_color=segment.local_scanned_area(Redundancy=False, Elimination=False)
coverage=coverage_rate(segment)
pass_points=o3d.geometry.PointCloud()
pass_points.points=o3d.utility.Vector3dVector(np.vstack(On_surface))
pass_points.colors=o3d.utility.Vector3dVector(np.vstack(On_surface_color))
revised_points=np.vstack(On_surface)
segment.mesh.vertex_colors = o3d.utility.Vector3dVector(segment.colors)
#o3d.visualization.draw_geometries([segment.mesh,pass_points],mesh_show_back_face=True)
smoothing_coeff=0.5

for i in range(10):
    old_points=revised_points.copy()

    #segment.passes=Potential_Field(segment, On_surface)
    segment.passes=LapReg_Potential_Field(segment, On_surface, smoothing_coeff, )
    #segment.passes=Learned_LapReg_Potential_Field(segment, On_surface, smoothing_coeff, coverage)

    redundant, On_surface, On_surface_color=segment.local_scanned_area(Redundancy=False, Elimination=False)

    coverage=coverage_rate(segment)


    pass_points=o3d.geometry.PointCloud()
    pass_points.points=o3d.utility.Vector3dVector(np.vstack(On_surface))
    pass_points.colors=o3d.utility.Vector3dVector(np.vstack(On_surface_color))
    revised_points=np.vstack(On_surface)
    segment.mesh.vertex_colors = o3d.utility.Vector3dVector(segment.colors)
    #o3d.visualization.draw_geometries([segment.mesh,pass_points],mesh_show_back_face=True)

    arrows = [segment.mesh, pass_points]
    for start, end in zip(old_points, revised_points):
        if np.linalg.norm(end - start) > 1e-3:  # Skip nearly-zero shifts
            arrows.append(create_arrow(start, end))
    o3d.visualization.draw_geometries(arrows)


Red: 31.285524247188295%, Green: 20.897327367275366%, Blue: 47.81714838553634%
Red: 29.350586527996132%, Green: 21.465715322288066%, Blue: 49.1836981497158%
Red: 28.141250453501026%, Green: 22.85645180795743%, Blue: 49.002297738541536%
Red: 27.040754625710484%, Green: 23.81182730680856%, Blue: 49.147418067480956%
Red: 25.891885354940136%, Green: 24.960696577578908%, Blue: 49.147418067480956%
Red: 24.440682065546014%, Green: 26.218406095053815%, Blue: 49.34091183940017%
Red: 23.848107389043417%, Green: 27.06494134720039%, Blue: 49.0869512637562%
Red: 22.082476720280567%, Green: 28.866852098198088%, Blue: 49.050671181521345%
Red: 21.501995404522916%, Green: 29.398959970975934%, Blue: 49.09904462450115%
Red: 19.905671786189384%, Green: 31.20087072197364%, Blue: 48.89345749183698%
Red: 18.877736122868548%, Green: 32.337646631999036%, Blue: 48.784617245132424%
